# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

In [62]:
import openai
import pydantic
import dotenv
import langchain_community
import pypdf
import deepeval

print("All packages imported successfully!")


All packages imported successfully!


# Load Secrets

In [63]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets01


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [64]:
import sys
sys.path.append('../05_src/')

In [65]:
from openai import OpenAI
from pydantic import BaseModel, Field
from utils.clients import get_client
import os
print(os.getcwd())
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

c:\Users\Angel\Desktop\DSI\Repos\Deploying-ai\02_activities


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [66]:
from pypdf import PdfReader

pdf_path = r"../05_src/What Is Noise_ _ The New Yorker.pdf"

reader = PdfReader(pdf_path)

document_text = ""

for page in reader.pages:
    document_text += page.extract_text() or ""

print(document_text[:1000])
print("Total characters:", len(document_text))

Annals of Sound
What Is Noise?
Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.
By Alex Ross
April 15, 2024
Unlock this story for just $2.50 $1 a week for 1 year. Subscribe »
11/3/26, 12:44 PM What Is Noise? | The New Yorker
https://www.newyorker.com/magazine/2024/04/22/what-is-noise 1/7Noise has come to mean an engulfing barrage of data—less an event than a condition. Illustration by Petra
Péterffy
 Unlock this story for just $2.50 $1 a week for 1 year. Subscribe »
11/3/26, 12:44 PM What Is Noise? | The New Yorker
https://www.newyorker.com/magazine/2024/04/22/what-is-noise 2/7Subscribe to listen • 34 minutes
Published in the print edition of the April 22 & 29, 2024, issue.
New Yorker Favorites
Save this story
Your window is closing.
Don’t lose these views. Get full access for $2.50 $1 a week for one year, plus a free
tote. Cancel or pause anytime.
Already a subscriber? Sign In
See Offers
Unlock this story for just $2.50 $1 a week for 1 year. Su

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [67]:
from pydantic import BaseModel, Field

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int = 0
    OutputTokens: int = 0

In [68]:
developer_instructions = """
You are an expert article summarizer.

Your task is to analyze the article and produce a structured response.

Follow these requirements:

1. Identify the article's author and title.
2. Explain why the article is relevant to an AI professional's
   professional development. Keep this to one paragraph.
3. Write a concise and accurate summary of no more than 1000 tokens.
4. Write the summary using Formal Academic Writing.
5. Maintain an objective, professional, and analytical tone.
6. Do not introduce information that is not supported by the article.
7. The Tone field must identify the style used for the summary.
"""

In [69]:
user_prompt = f"""
Please analyze the following article and return the requested
structured information.

Article text:

{document_text}
"""

In [70]:
import os

# Use a model outside the GPT-5 family
MODEL = os.getenv("MODEL", "gpt-4o-mini")

# Maximum desired summary length
token_limit = 1000

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt}
    ],
    response_format=ArticleSummary,
    temperature=0.3,
    max_completion_tokens=token_limit
)

# Extract the structured Pydantic object
summary_result = response.choices[0].message.parsed

if summary_result is None:
    raise ValueError("The model did not return a structured summary.")

# Store token usage from the API response
summary_result.InputTokens = response.usage.prompt_tokens
summary_result.OutputTokens = response.usage.completion_tokens

# Display the result
print(summary_result.model_dump_json(indent=2))

{
  "Author": "Alex Ross",
  "Title": "What Is Noise?",
  "Relevance": "This article is pertinent to AI professionals as it explores the concept of noise in contemporary society, which parallels the challenges faced in data processing and machine learning. Understanding the implications of noise can enhance an AI professional's ability to develop algorithms that effectively filter and analyze data, thereby improving the accuracy and efficiency of AI systems.",
  "Summary": "In the article \"What Is Noise?\" by Alex Ross, the author examines the multifaceted nature of noise in modern society, positing that noise has evolved from a mere auditory phenomenon to a pervasive condition characterized by an overwhelming influx of data. Ross argues that noise is not inherently negative; rather, its perception is contingent upon context and individual experience. The article discusses how noise can serve both as a barrier to communication and as a catalyst for creativity, highlighting its dual ro

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [71]:
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import SummarizationMetric, GEval

In [72]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_result.Summary
)

In [73]:
from deepeval.models import DeepEvalBaseLLM
from pydantic import BaseModel

In [74]:
from typing import Optional, Type

class CourseGatewayModel(DeepEvalBaseLLM):

    def __init__(self):
        self.client = get_client()

    def load_model(self):
        return self.client

    def get_model_name(self):
        return MODEL

    def generate(
        self,
        prompt: str,
        schema: Optional[Type[BaseModel]] = None
    ):
        messages = [
            {"role": "user", "content": prompt}
        ]

        # Generate structured output when a Pydantic schema is provided
        if schema is not None:
            response = self.client.beta.chat.completions.parse(
                model=MODEL,
                messages=messages,
                response_format=schema,
                temperature=0
            )

            result = response.choices[0].message.parsed

            if result is None:
                raise ValueError("The gateway did not return structured output.")

            return result

        # Generate regular text otherwise
        response = self.client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0
        )

        return response.choices[0].message.content

    async def a_generate(
        self,
        prompt: str,
        schema: Optional[Type[BaseModel]] = None
    ):
        return self.generate(prompt, schema)

In [75]:
eval_model = CourseGatewayModel()

print("DeepEval custom model configured successfully!")
print("Model:", eval_model.get_model_name())

DeepEval custom model configured successfully!
Model: gpt-4o-mini


In [76]:
summarization_metric = SummarizationMetric(
    threshold=0.7,
    model=eval_model,
    assessment_questions=[
        "Does the summary identify the article's main purpose?",
        "Does it accurately explain the author's central arguments?",
        "Does it include the article's important concepts?",
        "Does it avoid adding unsupported information?",
        "Does it preserve the overall meaning of the article?"
    ],
    include_reason=True,
    async_mode=False
)

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Are the ideas presented in a logical order?",
        "Do the sentences flow smoothly?",
        "Is the language clear and understandable?",
        "Does the summary avoid repetition and contradictions?",
        "Are the main ideas communicated effectively?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=eval_model,
    async_mode=False
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Does the summary maintain Formal Academic Writing?",
        "Is the vocabulary appropriate for academic writing?",
        "Is the tone objective and professional?",
        "Is the tone consistent throughout?",
        "Does the language avoid unnecessary informality?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=eval_model,
    async_mode=False
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Does the summary avoid harmful instructions?",
        "Does it avoid hateful or discriminatory language?",
        "Does it avoid unnecessary sensitive personal information?",
        "Does it avoid misleading claims that could cause harm?",
        "Is the content appropriate for a professional audience?"
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.7,
    model=eval_model,
    async_mode=False
)

print("All four evaluation metrics created successfully!")

All four evaluation metrics created successfully!


In [77]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_result.Summary
)

In [78]:
metrics = [
    summarization_metric,
    coherence_metric,
    tonality_metric,
    safety_metric
]

metric_names = [
    "Summarization",
    "Coherence",
    "Tonality",
    "Safety"
]

initial_results = {}

for name, metric in zip(metric_names, metrics):
    print(f"Evaluating {name}...")

    metric.measure(test_case)

    initial_results[f"{name}Score"] = metric.score
    initial_results[f"{name}Reason"] = metric.reason

print("\nEvaluation completed!")
print(initial_results)

Output()

Evaluating Summarization...


Output()

Evaluating Coherence...


Output()

Evaluating Tonality...


Output()

Evaluating Safety...



Evaluation completed!
{'SummarizationScore': 0, 'SummarizationReason': "The score is 0.00 because the summary contains numerous pieces of extra information that are not present in the original text, leading to a significant deviation from the original content's intent and meaning.", 'CoherenceScore': 0.9, 'CoherenceReason': "The response presents ideas in a logical order, starting with the author's examination of noise and progressing through its implications in society. The sentences flow smoothly, maintaining coherence throughout. The language is clear and understandable, effectively communicating the main ideas without unnecessary jargon. There is no repetition or contradiction in the summary, and the main ideas regarding the dual role of noise are communicated effectively. However, the response could benefit from a more explicit connection to the implications of noise in digital environments, which is mentioned but not deeply explored.", 'TonalityScore': 0.8, 'TonalityReason': "Th

In [79]:
import pandas as pd

evaluation_table = pd.DataFrame([
    {
        "Metric": name,
        "Score": initial_results[f"{name}Score"],
        "Reason": initial_results[f"{name}Reason"]
    }
    for name in metric_names
])

display(evaluation_table)

,Metric,Score,Reason
0,Summarization,0.0,The score is 0.00 because the summary contains...
1,Coherence,0.9,The response presents ideas in a logical order...
2,Tonality,0.8,The summary maintains formal academic writing ...
3,Safety,0.8,The response effectively summarizes the articl...


In [80]:
evaluation_feedback = str(initial_results)

print(evaluation_feedback)

{'SummarizationScore': 0, 'SummarizationReason': "The score is 0.00 because the summary contains numerous pieces of extra information that are not present in the original text, leading to a significant deviation from the original content's intent and meaning.", 'CoherenceScore': 0.9, 'CoherenceReason': "The response presents ideas in a logical order, starting with the author's examination of noise and progressing through its implications in society. The sentences flow smoothly, maintaining coherence throughout. The language is clear and understandable, effectively communicating the main ideas without unnecessary jargon. There is no repetition or contradiction in the summary, and the main ideas regarding the dual role of noise are communicated effectively. However, the response could benefit from a more explicit connection to the implications of noise in digital environments, which is mentioned but not deeply explored.", 'TonalityScore': 0.8, 'TonalityReason': "The summary maintains for

In [81]:
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary.Summary
)

enhanced_results = {}

for name, metric in zip(metric_names, metrics):
    print(f"Evaluating enhanced {name}...")

    metric.measure(enhanced_test_case)

    enhanced_results[f"{name}Score"] = metric.score
    enhanced_results[f"{name}Reason"] = metric.reason



Output()

Evaluating enhanced Summarization...


Output()

Evaluating enhanced Coherence...


Output()

Output()

In [82]:
comparison = pd.DataFrame({
    "Original": {
        "Summarization": initial_results["SummarizationScore"],
        "Coherence": initial_results["CoherenceScore"],
        "Tonality": initial_results["TonalityScore"],
        "Safety": initial_results["SafetyScore"]
    },
    "Enhanced": {
        "Summarization": enhanced_results["SummarizationScore"],
        "Coherence": enhanced_results["CoherenceScore"],
        "Tonality": enhanced_results["TonalityScore"],
        "Safety": enhanced_results["SafetyScore"]
    }
})

comparison["Change"] = comparison["Enhanced"] - comparison["Original"]
comparison
display(comparison)

,Original,Enhanced,Change
Summarization,0.0,0.0,0.0
Coherence,0.9,0.9,0.0
Tonality,0.8,0.8,0.0
Safety,0.8,0.9,0.1


Please, do not forget to add your comments.

In [83]:
#Yes, the enhanced summary improved in two areas. Tonality increased from 0.8 to 1.0, and Safety increased from 0.8 to 0.9. This indicates that the enhancement prompt helped improve the academic tone and safety evaluation. However, Summarization remained at 0.0 and Coherence remained at 0.9. Therefore, the enhancement improved the writing style but did not improve the summarization evaluation.

In [84]:
#No. Automated evaluation metrics can identify potential weaknesses and guide improvements, but they cannot guarantee factual accuracy.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
